# Play Pacman With Coins

Choose an environment with the buttons, preview an `rgb_array` frame inline, then run the last cell to play with the arrow keys in a pygame window. Set `RECORD_VIDEO = True` to save the environment canvas as a 1920x1080 MP4.

In [ ]:
import ipywidgets as widgets
from IPython.display import display
from PIL import Image
from masa.common.rendering.notebook_play import make_reset_env, sync_selected_env
from masa.common.rendering.notebook_play import start_recorded_play_thread

RECORD_VIDEO = True
VIDEO_PATH = "videos/notebooks/play_pacman_coins.mp4"
VALID_ENV_NAMES = ("MiniPacmanWithCoins", "PacmanWithCoins")

def make_env(env_name, **kwargs):
    if env_name == "MiniPacmanWithCoins":
        from masa.envs.discrete.mini_pacman_with_coins import MiniPacmanWithCoins
        return MiniPacmanWithCoins(**kwargs)
    if env_name == "PacmanWithCoins":
        from masa.envs.discrete.pacman_with_coins import PacmanWithCoins
        return PacmanWithCoins(**kwargs)
    raise ValueError(f"env_name must be one of {VALID_ENV_NAMES!r}")

ENV_SELECTOR = widgets.ToggleButtons(options=VALID_ENV_NAMES, value="MiniPacmanWithCoins", description="Env")
display(ENV_SELECTOR)

ENV_NAME = ENV_SELECTOR.value
PACMAN_HAT = "crown"
GHOST_COLORS = ((71, 202, 255),)
SEED = 0


In [ ]:
ENV_NAME = ENV_SELECTOR.value
preview_env = make_env(
    ENV_NAME, render_mode="rgb_array", render_window_size=512,
    pacman_hat=PACMAN_HAT, ghost_colors=GHOST_COLORS,
)
preview_env.reset(seed=SEED)
frame = preview_env.render()
display(Image.fromarray(frame))
preview_env.close()


In [ ]:
def _make_env_kwargs():
    return {"pacman_hat": PACMAN_HAT, "ghost_colors": GHOST_COLORS}

def play(env_name=None, seed=SEED):
    follow_selector = env_name is None

    def _run(stop_event):
        import pygame
        selected_env_name = env_name
        if follow_selector:
            selected_env_name = ENV_SELECTOR.value
        action_keys = {pygame.K_LEFT: 0, pygame.K_RIGHT: 1, pygame.K_DOWN: 2, pygame.K_UP: 3, pygame.K_SPACE: 4}
        env, _, _ = make_reset_env(
            make_env, selected_env_name, seed=seed, render_mode="human", render_window_size=512,
            env_kwargs=_make_env_kwargs(),
        )
        try:
            running = True
            while running and not stop_event.is_set() and not env.human_window_closed:
                if follow_selector:
                    env, selected_env_name, _, _, switched = sync_selected_env(
                        env, selected_env_name, ENV_SELECTOR, make_env, seed=seed, render_mode="human",
                        render_window_size=512, env_kwargs=_make_env_kwargs(), pygame=pygame,
                    )
                    if switched:
                        print("switched:", selected_env_name)
                chosen_action = None
                for event in pygame.event.get():
                    if not env.handle_pygame_event(event):
                        running = False
                        break
                    if event.type == pygame.KEYDOWN:
                        if event.key in (pygame.K_q, pygame.K_ESCAPE):
                            running = False
                            break
                        chosen_action = action_keys.get(event.key, chosen_action)
                if not running or stop_event.is_set():
                    break
                if chosen_action is None:
                    env.render()
                    continue
                _, reward, terminated, truncated, _ = env.step(chosen_action)
                if reward:
                    print(f"reward={reward}")
                if terminated or truncated:
                    env.reset(seed=seed)
        finally:
            env.close()

    return start_recorded_play_thread("pacman_coins", _run, record_video=RECORD_VIDEO, video_path=VIDEO_PATH)

play_session = play()
